In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, roc_auc_score
from imblearn.over_sampling import SMOTE
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

In [ ]:
# 1. Load dataset
df = pd.read_csv("loan_data.csv")
df.columns

Index(['credit.policy', 'purpose', 'int.rate', 'installment', 'log.annual.inc',
       'dti', 'fico', 'days.with.cr.line', 'revol.bal', 'revol.util',
       'inq.last.6mths', 'delinq.2yrs', 'pub.rec', 'not.fully.paid'],
      dtype='object')

In [7]:
# 2. Encode categorical features
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

In [8]:
# 3. Features and target
X = df.drop('not.fully.paid', axis=1)
y = df['not.fully.paid']

In [9]:
# 4. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

In [10]:
# 5. Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [11]:
# 7. Handle imbalance with SMOTE
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

In [12]:
# 8. Build deep learning model
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=[tf.keras.metrics.Precision(name='precision'),
                       tf.keras.metrics.Recall(name='recall'),
                       tf.keras.metrics.AUC(name='auc')])

C:\Users\keert\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [13]:
# 9. Train model
history = model.fit(X_train, y_train,
                    validation_data=(X_test, y_test),
                    epochs=10,
                    batch_size=256,
                    verbose=1)


Epoch 1/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 6s 33ms/step - auc: 0.6310 - loss: 0.6666 - precision: 0.5893 - recall: 0.6010 - val_auc: 0.6589 - val_loss: 0.6116 - val_precision: 0.2450 - val_recall: 0.5065
Epoch 2/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - auc: 0.6669 - loss: 0.6485 - precision: 0.6194 - recall: 0.6029 - val_auc: 0.6670 - val_loss: 0.6339 - val_precision: 0.2275 - val_recall: 0.5761
Epoch 3/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - auc: 0.6827 - loss: 0.6390 - precision: 0.6323 - recall: 0.6445 - val_auc: 0.6677 - val_loss: 0.6185 - val_precision: 0.2387 - val_recall: 0.5717
Epoch 4/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - auc: 0.6881 - loss: 0.6345 - precision: 0.6282 - recall: 0.6356 - val_auc: 0.6668 - val_loss: 0.6359 - val_precision: 0.2333 - val_recall: 0.6087
Epoch 5/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - auc: 0.6948 - loss: 0.6293 - precision: 0.6335 - recall: 0.6455 - val_auc: 0.6675 - val_loss: 0.6343 - val_precision: 0.2320 - val_recall: 0.6022
Epoch 